# 실습 2주차: 퍼셉트론을 쌓아 신경망 만들기

> **시나리오 — 오늘 만들 것**
>
>
> 자동차 **392대의 마력(horsepower)** 으로 **연비(mpg)** 를 맞힌다.
>
> 1주차 모형은 **직선**이었다. 그런데 이 관계는 직선이 아니다 —
> 마력이 낮을 때는 연비가 가파르게 떨어지다가, 높아지면 완만해진다.
>
> **직선으로 안 되는 것을 층을 쌓아 푼다.** 세 모형을 학습시켜 나란히 비교한다.
>
> 1. 퍼셉트론 하나 (= 1주차 모형)
> 2. 두 층인데 **활성화 함수가 없는** 모형
> 3. 두 층 + **ReLU**
>
> - **대응 이론**: [Ch02 퍼셉트론과 다층 퍼셉트론](ch02.qmd)


> **오늘 배우는 PyTorch 부품**
>
>
> | 부품 | 이론에서의 이름 |
> |------|------|
> | `nn.Linear` | **퍼셉트론 / 층** — 1주차에 직접 만든 그것 |
> | `nn.ReLU` · `nn.Sigmoid` · `nn.Tanh` | 활성화 함수 |
> | `nn.Sequential` | 층을 이어 붙이기 |
> | `DataLoader` | 데이터셋에서 데이터를 꺼내 오는 장치 |
> | `torch.optim.SGD` + 학습 루프 4줄 | 파라미터를 고쳐 나가는 표준 절차 |
>
> 마지막 줄은 **오늘 뜻을 배우지 않는다.** 3주차에 한 줄씩 뜯어본다.
> 오늘은 "층을 쌓으면 무엇이 달라지는가"만 본다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

---

# 1. 직선으로 안 된다는 것을 눈으로 본다

## 1-1. 데이터 불러오기

In [ ]:
URL = 'https://raw.githubusercontent.com/ralbu85/Lecture_DeepLearning_2022/main/auto.csv'
a = pd.read_csv(URL)
print(a.shape)
a.head()

## 1-2. 마력과 연비의 관계

In [ ]:
plt.figure(figsize=(5.6, 3.8))
plt.scatter(a['horsepower'], a['mpg'], s=14, alpha=0.6)
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**휘어 있다.** 마력 50~100 구간에서는 급하게 떨어지고, 150 이상에서는 거의 평평하다.

## 1-3. 직선을 그어 보면

In [ ]:
from sklearn.linear_model import LinearRegression

Xn = a[['horsepower']].to_numpy(dtype='float32')
yn = a['mpg'].to_numpy(dtype='float32')

line = LinearRegression().fit(Xn, yn)
grid = np.linspace(Xn.min(), Xn.max(), 200).reshape(-1, 1).astype('float32')

plt.figure(figsize=(5.6, 3.8))
plt.scatter(Xn, yn, s=14, alpha=0.5, label='data')
plt.plot(grid, line.predict(grid), 'r-', lw=2, label='straight line')
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

양쪽 끝에서 **체계적으로 빗나간다.** 직선이 표현할 수 있는 모양의 한계다.

> **직접 해보기 ① — 다른 변수도 휘어 있는가**
>
>
> `weight` 와 `mpg` 의 산점도를 그려 보시오.

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(5.6, 3.8))
plt.scatter(...)                  # ← 여기를 채우세요
plt.xlabel('weight'); plt.ylabel('mpg')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(5.6, 3.8))
plt.scatter(a['weight'], a['mpg'], s=14, alpha=0.6, color='C1')
plt.xlabel('weight'); plt.ylabel('mpg')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---

# 2. 퍼셉트론 하나 = `nn.Linear`

## 2-1. 지난주의 `X @ w + b` 가 곧 퍼셉트론이다

1주차 마지막에 이렇게 예측했다.

```python
yhat = X @ w + b        # (n, p) @ (p,) → (n,)
```

**이것이 퍼셉트론 하나**다. 가중치를 곱해 더하고 편향을 더하는 것 —
Ch02의 퍼셉트론 정의 그대로다. 그리고 PyTorch에는 이것을 담은 부품이 이미 있다.

In [ ]:
layer = nn.Linear(in_features=3, out_features=1)

print(layer)
for name, param in layer.named_parameters():
    print(f'  {name:7s} {tuple(param.shape)}')
print('파라미터 수:', sum(p.numel() for p in layer.parameters()), '= 3 x 1 + 1')

> **`nn.Linear` 의 가중치 모양은 `(출력, 입력)` 이다**
>
>
> 1주차에는 `w` 를 `(p,)` 로 두고 `X @ w` 를 했다.
> `nn.Linear` 는 가중치를 **`(출력, 입력)`** 으로 두고 $xW^\top + b$ 를 계산한다.
>
> $$\texttt{nn.Linear(3, 1).weight.shape} \;=\; (1,\ 3)$$
>
> **결과는 같다.** 순서를 헷갈리면 shape 에러가 나므로 항상 찍어 본다.


## 2-2. 손계산과 대조

$$z = w_1 x_1 + w_2 x_2 + w_3 x_3 + b$$

In [ ]:
x = np.array([1.0, 2.0, 3.0], dtype='float32')
w = np.array([0.5, -1.0, 2.0], dtype='float32')
b = 0.1

z = (w * x).sum() + b               # 곱해서 더하고, 편향을 더한다
print('손계산 z =', round(float(z), 4), '  = 0.5x1 + (-1)x2 + 2x3 + 0.1')

with torch.no_grad():               # 우리 숫자를 그대로 넣어 본다
    layer.weight.copy_(torch.tensor([w]))
    layer.bias.copy_(torch.tensor([b]))

print('nn.Linear:', round(float(layer(torch.tensor(x))), 4))

## 2-3. 여러 대를 한 번에

In [ ]:
X5 = torch.randn(5, 3)
print('입력 :', tuple(X5.shape))
print('출력 :', tuple(layer(X5).shape), '  ← 대당 하나씩')
print(layer(X5).detach().numpy().round(3))

> **직접 해보기 ② — 입력 4개짜리 퍼셉트론**
>
>
> 입력 4개, 출력 1개인 퍼셉트론을 만들고 자동차 10대짜리 입력을 통과시키시오.

In [ ]:
# ✏️ 직접 채워 보세요
p = None                    # ← nn.Linear(...)
X10 = torch.randn(10, 4)
out10 = None                # ← p(X10)

assert out10 is not None and tuple(out10.shape) == (10, 1), '모양을 확인하세요'
print('통과  출력', tuple(out10.shape),
      ' 파라미터', sum(q.numel() for q in p.parameters()))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
p = nn.Linear(4, 1)
X10 = torch.randn(10, 4)
out10 = p(X10)
print('출력', tuple(out10.shape),
      ' 파라미터', sum(q.numel() for q in p.parameters()), '= 4 x 1 + 1')

---

# 3. 노드를 여러 개로 — 층

퍼셉트론 하나는 숫자 하나를 낸다. **여러 개를 나란히 놓은 것**이 층이다.

In [ ]:
hidden = nn.Linear(1, 8)          # 입력 1개 → 노드 8개

print('weight :', tuple(hidden.weight.shape), ' = (출력, 입력)')
print('bias   :', tuple(hidden.bias.shape))
print('파라미터:', sum(q.numel() for q in hidden.parameters()), ' = 1 x 8 + 8')

x1 = torch.tensor([[100.0]])      # 마력 100인 자동차 한 대
print('\n입력:', tuple(x1.shape), '→ 출력:', tuple(hidden(x1).shape))
print(hidden(x1).detach().numpy().round(3))

노드 8개가 **같은 입력을 서로 다른 가중치로** 본다. 8개의 서로 다른 관점이 생긴 것이다.

> **직접 해보기 ③ — 파라미터 수를 먼저 예측하기**
>
>
> `nn.Linear(5, 12)` 의 파라미터 수는? **먼저 계산한 뒤** 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
my_answer = None            # ← 예상한 숫자

real = sum(q.numel() for q in nn.Linear(5, 12).parameters())
assert my_answer == real, f'다릅니다. 실제는 {real} — 공식은 (입력 x 출력) + 출력'
print('정답', real)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
my_answer = 5 * 12 + 12
real = sum(q.numel() for q in nn.Linear(5, 12).parameters())
print('예상', my_answer, ' 실제', real, ' 공식 (입력 x 출력) + 출력')

---

# 4. 활성화 함수 — 층을 쌓는 의미를 만드는 것

## 4-1. 원소마다 적용된다

In [ ]:
z = torch.tensor([[-2.0, -0.5, 0.0, 1.5, 3.0]])

print('입력    :', z.numpy()[0])
print('ReLU    :', nn.ReLU()(z).numpy()[0])
print('Sigmoid :', nn.Sigmoid()(z).numpy()[0].round(4))
print('Tanh    :', nn.Tanh()(z).numpy()[0].round(4))
print('\n파라미터 수:', sum(q.numel() for q in nn.ReLU().parameters()), ' ← 배우는 것이 없다')

In [ ]:
t = torch.linspace(-4, 4, 200)
plt.figure(figsize=(6, 3.2))
for f, name in [(nn.ReLU(), 'ReLU'), (nn.Sigmoid(), 'Sigmoid'), (nn.Tanh(), 'Tanh')]:
    plt.plot(t, f(t), label=name)
plt.axhline(0, lw=0.6); plt.axvline(0, lw=0.6)
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4-2. 활성화가 없으면 층을 쌓아도 층 하나다

In [ ]:
torch.manual_seed(0)
L1 = nn.Linear(3, 8, bias=False)
L2 = nn.Linear(8, 2, bias=False)

Xc = torch.randn(4, 3)
two_layers = L2(L1(Xc))                      # 두 층을 통과

W_combined = L2.weight @ L1.weight           # (2,8) @ (8,3) → (2,3)
one_layer = Xc @ W_combined.T                # 층 하나로 같은 계산

print('두 층 통과:\n', two_layers.detach().numpy().round(4))
print('\n층 하나로 :\n', one_layer.detach().numpy().round(4))
print('\n같은가:', torch.allclose(two_layers, one_layer, atol=1e-5))
print('합쳐진 가중치 shape:', tuple(W_combined.shape), ' ← 결국 (3 → 2) 한 층')

> **이것이 활성화 함수가 필요한 이유**
>
>
> $$W_2(W_1 x) = (W_2 W_1) x$$
>
> 행렬 두 개를 곱하면 **행렬 하나**다. 활성화 없이 층을 100개 쌓아도 결국 **직선 하나**다.
> 중간에 ReLU처럼 **직선이 아닌 함수**를 끼워야 비로소 층이 층 노릇을 한다.


> **직접 해보기 ④ — ReLU를 넣으면 깨지는가**
>
>
> 두 층 사이에 `nn.ReLU()` 를 넣고, 여전히 `Xc @ W_combined.T` 와 같은지 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
with_relu = None            # ← L2(nn.ReLU()(L1(Xc)))

print('층 하나와 같은가:', torch.allclose(with_relu, one_layer, atol=1e-5))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
with_relu = L2(nn.ReLU()(L1(Xc)))
print('ReLU 넣은 결과:\n', with_relu.detach().numpy().round(4))
print('층 하나와 같은가:', torch.allclose(with_relu, one_layer, atol=1e-5), ' ← 더 이상 같지 않다')

---

# 5. 층 쌓기 — `nn.Sequential`

In [ ]:
mlp = nn.Sequential(
    nn.Linear(1, 16),      # 마력 1개 → 은닉 노드 16개
    nn.ReLU(),
    nn.Linear(16, 1),      # 은닉 16개 → 연비 1개
)
print(mlp)

In [ ]:
h = torch.zeros(5, 1)
for i, lay in enumerate(mlp):
    h = lay(h)
    print(f'{i} {lay.__class__.__name__:8s} → {tuple(h.shape)}  '
          f'파라미터 {sum(q.numel() for q in lay.parameters())}')
print('\n전체 파라미터:', sum(q.numel() for q in mlp.parameters()), ' = (1x16+16) + (16x1+1)')

---

# 6. 완성 — 세 모형을 학습시켜 비교한다

## 6-1. 데이터셋과 데이터로더

1주차의 `TensorDataset` 에 **`DataLoader`** 를 붙인다.
`Dataset` 이 "무엇이 있는가"라면 `DataLoader` 는 **"어떻게 꺼내는가"** 다.

In [ ]:
X = torch.tensor(Xn)
y = torch.tensor(yn).unsqueeze(1)

# 숫자 크기를 맞춰 준다 — 왜 필요한지는 4주차에 배운다
mu, sd = X.mean(0), X.std(0)
ym, ys = y.mean(), y.std()
Xs = (X - mu) / sd
ys_ = (y - ym) / ys

ds = TensorDataset(Xs, ys_)
loader = DataLoader(ds, batch_size=len(ds))    # 오늘은 전부 한 번에 꺼낸다

print('데이터 수      :', len(ds))
print('에폭당 배치 수 :', len(loader))
xb, yb = next(iter(loader))
print('한 배치        :', tuple(xb.shape), tuple(yb.shape))

> `batch_size` 를 데이터 전체 크기로 두었으므로 한 번에 다 꺼내진다.
> **몇 건씩 꺼낼지**가 학습에 어떤 영향을 주는지는 **4주차**에서 다룬다.
> 로더를 쓰는 이 형태는 15주 내내 바뀌지 않는다.


## 6-2. 표준 학습 루프

In [ ]:
def train(model, loader, epochs=1000, lr=0.05):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()              # ①
            loss = criterion(model(xb), yb)    # ②
            loss.backward()                    # ③
            optimizer.step()                   # ④
            history.append(loss.item())
    return history

> **이 네 줄은 오늘 **이해하지 않는다****
>
>
> ①~④가 각각 무슨 뜻인지는 **3주차에 한 줄씩** 배운다.
> `zero_grad` 가 왜 필요한지, `backward()` 가 무엇을 계산하는지, `step()` 이 무엇을 바꾸는지 —
> 전부 3주차의 내용이다.
>
> 오늘은 이 네 줄을 **그대로 옮겨 적고**, 결과만 본다.
> 숨겨 놓은 함수가 아니라 **앞으로 계속 직접 쓸 네 줄**이므로 눈에 익혀 두면 된다.


## 6-3. 세 모형을 학습시킨다

In [ ]:
def rmse_of(model):
    with torch.no_grad():
        pred = model(Xs).squeeze(1) * ys + ym          # 원래 단위(mpg)로 되돌린다
    return float(((pred - y.squeeze(1)) ** 2).mean().sqrt())

models = {
    '1층 (선형)':      lambda: nn.Linear(1, 1),
    '2층 활성화 없음': lambda: nn.Sequential(nn.Linear(1, 16), nn.Linear(16, 1)),
    '2층 + ReLU':      lambda: nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1)),
}

trained, hists, rows = {}, {}, []
for name, make in models.items():
    torch.manual_seed(42)
    m = make()
    hists[name] = train(m, loader)
    trained[name] = m
    rows.append({'모형': name,
                 '파라미터': sum(q.numel() for q in m.parameters()),
                 'RMSE (mpg)': round(rmse_of(m), 3)})

print(pd.DataFrame(rows).to_string(index=False))

> **앞의 두 줄을 보라 — 숫자가 **똑같다****
>
>
> 파라미터는 2개와 49개로 24배 차이인데 성능은 **소수점까지 같다.**
> 4-2절에서 증명한 그대로 — 활성화 없는 2층은 **선형 모형 그 자체**다.


## 6-4. 손실이 내려가는 모습

In [ ]:
plt.figure(figsize=(6.2, 3.6))
for name in models:
    plt.plot(hists[name], label=name)
plt.xlabel('epoch'); plt.ylabel('MSE (standardized)')
plt.yscale('log'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6-5. 무엇을 그렸는지 눈으로 확인한다

In [ ]:
gt = torch.tensor((grid - mu.numpy()) / sd.numpy())

plt.figure(figsize=(6.4, 4.2))
plt.scatter(Xn, yn, s=14, alpha=0.35, color='gray', label='data')
for name, m in trained.items():
    with torch.no_grad():
        curve = m(gt).squeeze(1).numpy() * float(ys) + float(ym)
    plt.plot(grid, curve, lw=2, label=name)
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

앞의 두 모형은 **직선 하나로 겹쳐 있고**, ReLU를 넣은 모형만 **꺾인 선**으로 데이터를 따라간다.

> ReLU는 꺾인 직선이다. 그것을 16개 겹치면 **꺾은선으로 곡선을 흉내낼 수 있다.**
> 은닉 노드를 늘릴수록 더 촘촘한 꺾은선이 된다.


> **직접 해보기 ⑤ — 은닉 노드 수를 바꿔 보기**
>
>
> 은닉 노드를 `2, 4, 16, 64` 로 바꿔 학습시키고 예측 곡선을 한 그림에 겹쳐 보시오.
> 노드가 적으면 곡선이 어떻게 되는가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(6.4, 4.2))
plt.scatter(Xn, yn, s=14, alpha=0.3, color='gray')
for hsize in [2, 4, 16, 64]:
    torch.manual_seed(42)
    m = ...                       # ← 은닉 hsize개짜리 nn.Sequential
    train(m, loader)
    with torch.no_grad():
        plt.plot(grid, m(gt).squeeze(1).numpy() * float(ys) + float(ym),
                 lw=2, label=f'hidden {hsize}')
plt.xlabel('horsepower'); plt.ylabel('mpg'); plt.legend(fontsize=8); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(6.4, 4.2))
plt.scatter(Xn, yn, s=14, alpha=0.3, color='gray')
for hsize in [2, 4, 16, 64]:
    torch.manual_seed(42)
    m = nn.Sequential(nn.Linear(1, hsize), nn.ReLU(), nn.Linear(hsize, 1))
    train(m, loader)
    with torch.no_grad():
        plt.plot(grid, m(gt).squeeze(1).numpy() * float(ys) + float(ym),
                 lw=2, label=f'hidden {hsize}  (RMSE {rmse_of(m):.2f})')
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

노드가 2개면 꺾이는 곳이 하나뿐이라 곡선을 따라가지 못한다.
**은닉 노드 수는 모형이 표현할 수 있는 모양의 복잡도**를 정한다 — 하이퍼파라미터다.

---

# 7. 정리

> **이번 주 체크포인트**
>
>
> | 하고 싶은 일 | 코드 | 결과 |
> |------|------|------|
> | 퍼셉트론 하나 | `nn.Linear(p, 1)` | 파라미터 $p+1$ |
> | 층 (노드 여러 개) | `nn.Linear(p, h)` | 파라미터 $p\cdot h + h$ |
> | 가중치 모양 확인 | `layer.weight.shape` | `(출력, 입력)` |
> | 활성화 | `nn.ReLU()`, `nn.Sigmoid()`, `nn.Tanh()` | 파라미터 0 |
> | 층 쌓기 | `nn.Sequential(...)` | |
> | 층별 shape 확인 | `for lay in model:` 하나씩 통과 | |
> | 데이터 꺼내기 | `DataLoader(ds, batch_size=...)` | `for xb, yb in loader` |
> | 학습 (3주차에 이해) | `zero_grad → loss → backward → step` | |


**오늘의 결론**

$$W_2(W_1x) = (W_2W_1)x \quad\Longrightarrow\quad \text{활성화가 없으면 층을 쌓아도 직선}$$

## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
net = nn.Sequential(
    nn.Linear(4, 10), nn.ReLU(),
    nn.Linear(10, 6), nn.ReLU(),
    nn.Linear(6, 3),
)
xq = torch.randn(7, 4)

h = xq
for i, lay in enumerate(net):
    h = lay(h)
    print(f'{i} {lay.__class__.__name__:8s} → {tuple(h.shape)}')

print('\n전체 파라미터:', sum(q.numel() for q in net.parameters()))
print('손으로:', (4*10+10), '+', (10*6+6), '+', (6*3+3), '=', (4*10+10)+(10*6+6)+(6*3+3))

dl = DataLoader(TensorDataset(xq, torch.randn(7, 3)), batch_size=3)
print('\n에폭당 배치 수:', len(dl))
for k, (bx, by) in enumerate(dl):
    print(f'  배치 {k}: {tuple(bx.shape)}')

---

## 다음 실습

[실습 3주차: 학습 루프를 직접 만든다](lab03.qmd) —
오늘 그대로 옮겨 적은 **네 줄의 뜻**을 한 줄씩 배운다.
손실 함수가 무엇이고, `backward()` 가 무엇을 계산하며, `step()` 이 무엇을 바꾸는지 연다.